Install Dependencies and Check GPU

In [1]:
!pip install -q transformers==4.44.2 accelerate einops timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 636.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.2 MB/s eta 0:00:00


Find all images

In [2]:
import os
print("Available datasets:")
for item in os.listdir("/kaggle/input/datasets/narmeensabahsiddiqui"):
    print(f"  {item}")

Available datasets:
  cityscapes-leftimg8bit-traintestval


In [3]:
import os
import glob

DATASET_NAME = "cityscapes-leftimg8bit-traintestval"
BASE_PATH = f"/kaggle/input/datasets/narmeensabahsiddiqui/{DATASET_NAME}"

# Find the training images
# Kaggle may or may not have an extra folder level depending on how the zip was structured
# Let's search broadly and see what we find
all_pngs = glob.glob(os.path.join(BASE_PATH, "**", "*_leftImg8bit.png"), recursive=True)
train_pngs = [p for p in all_pngs if "/train/" in p]
test_pngs = [ptst for ptst in all_pngs if "/test/" in ptst]
val_pngs = [pval for pval in all_pngs if "/val/" in pval]
print(f"Total PNGs found: {len(all_pngs)}")
print(f"Training PNGs found: {len(train_pngs)}")
print(f"Testing PNGs found: {len(test_pngs)}")
print(f"Validation PNGs found: {len(val_pngs)}")

if len(train_pngs) > 0:
    print(f"\nFirst 3 paths:")
    for p in train_pngs[:3]:
        print(f"  {p}")
    # Extract the common root for later use
    TRAIN_DIR = train_pngs[0].split("/train/")[0] + "/train"
    print(f"\nTraining directory: {TRAIN_DIR}")
else:
    # If nothing found, let's see what's actually in the dataset
    print(f"\nContents of {BASE_PATH}:")
    for item in os.listdir(BASE_PATH)[:20]:
        print(f"  {item}")
    print("\nTrain not found. Check the path and adjust DATASET_NAME accordingly")

if len(test_pngs) > 0:
    print(f"\nFirst 3 paths, testing:")
    for p in test_pngs[:3]:
        print(f"  {p}")
    # Extract the common root for later use
    TEST_DIR = test_pngs[0].split("/test/")[0] + "/test"
    print(f"\nTesting directory: {TEST_DIR}")
else:
    # If nothing found, let's see what's actually in the dataset
    print(f"\nContents of {BASE_PATH}:")
    for item in os.listdir(BASE_PATH)[:20]:
        print(f"  {item}")
    print("\nTest not found. Check the path and adjust DATASET_NAME accordingly")
if len(val_pngs) > 0:
    print(f"\nFirst 3 paths, validation:")
    for p in val_pngs[:3]:
        print(f"  {p}")
    # Extract the common root for later use
    VAL_DIR = val_pngs[0].split("/val/")[0] + "/val"
    print(f"\n Validation directory: {VAL_DIR}")
else:
    # If nothing found, let's see what's actually in the dataset
    print(f"\nContents of {BASE_PATH}:")
    for item in os.listdir(BASE_PATH)[:20]:
        print(f"  {item}")
    print("\nVal not found. Check the path and adjust DATASET_NAME accordingly")

Total PNGs found: 5000
Training PNGs found: 2975
Testing PNGs found: 1525
Validation PNGs found: 500

First 3 paths:
  /kaggle/input/datasets/narmeensabahsiddiqui/cityscapes-leftimg8bit-traintestval/leftImg8bit/train/dusseldorf/dusseldorf_000180_000019_leftImg8bit.png
  /kaggle/input/datasets/narmeensabahsiddiqui/cityscapes-leftimg8bit-traintestval/leftImg8bit/train/dusseldorf/dusseldorf_000083_000019_leftImg8bit.png
  /kaggle/input/datasets/narmeensabahsiddiqui/cityscapes-leftimg8bit-traintestval/leftImg8bit/train/dusseldorf/dusseldorf_000096_000019_leftImg8bit.png

Training directory: /kaggle/input/datasets/narmeensabahsiddiqui/cityscapes-leftimg8bit-traintestval/leftImg8bit/train

First 3 paths, testing:
  /kaggle/input/datasets/narmeensabahsiddiqui/cityscapes-leftimg8bit-traintestval/leftImg8bit/test/mainz/mainz_000000_013437_leftImg8bit.png
  /kaggle/input/datasets/narmeensabahsiddiqui/cityscapes-leftimg8bit-traintestval/leftImg8bit/test/mainz/mainz_000003_008690_leftImg8bit.png
 

Load Florence 2

In [4]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import os
from unittest.mock import patch
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports

MODEL_ID = "microsoft/Florence-2-large"

# Workaround: Florence-2's code requires flash_attn but it's not actually needed
# The following patch removes flash_attn from the import list before the model loads
def fixed_get_imports(filename):
    if not str(filename).endswith("modeling_florence2.py"):
        return get_imports(filename)
    imports = get_imports(filename)
    if "flash_attn" in imports:
        imports.remove("flash_attn")
    return imports

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        attn_implementation="sdpa"  # use PyTorch's built-in efficient attention instead
    ).to("cuda").eval()

print("Florence-2-large loaded successfully")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU available: True
GPU: Tesla T4
VRAM: 15.6 GB


2026-04-23 16:39:41.978464: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776962382.133198      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776962382.178915      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776962382.542229      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776962382.542262      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776962382.542265      23 computation_placer.cc:177] computation placer alr

preprocessor_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

processing_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

modeling_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

Florence-2-large loaded successfully
GPU memory used: 1.55 GB


In [5]:
import json
from PIL import Image
from tqdm import tqdm
import re

TASK = "<DETAILED_CAPTION>"
OUTPUT_DIR = "/kaggle/working/"
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "_checkpoint.json")
CHECKPOINT_EVERY = 100

# Load checkpoint if exists (for resume)
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH, "r") as f:
        results = json.load(f)
    done_files = {r["file_name"] for r in results}
    print(f"Resuming from checkpoint: {len(results)} already captioned")
else:
    results = []
    done_files = set()

# Sort for reproducibility
train_image_paths = sorted(train_pngs)
test_image_paths = sorted(test_pngs)
val_image_paths = sorted(val_pngs)

for img_path in tqdm(train_image_paths, desc="Captioning"):
    # Use relative path from the train directory as the identifier
    rel_path = os.path.relpath(img_path, os.path.dirname(TRAIN_DIR))
    
    if rel_path in done_files:
        continue
    
    image = Image.open(img_path).convert("RGB")
    inputs = processor(text=TASK, images=image, return_tensors="pt").to("cuda", torch.float16)
    
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1024,
            do_sample=False,
            num_beams=3
        )
    
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed = processor.post_process_generation(
        generated_text, task=TASK, image_size=(image.width, image.height)
    )
    caption = parsed[TASK]

    caption = re.sub(r'^The image shows\s+', '', caption, flags=re.IGNORECASE)
    caption = re.sub(r'^The image depicts\s+', '', caption, flags=re.IGNORECASE)
    caption = re.sub(r'^The image features\s+', '', caption, flags=re.IGNORECASE)
    caption = re.sub(r'^This image shows\s+', '', caption, flags=re.IGNORECASE)
    # Capitalize the first letter after stripping
    caption = caption[0].upper() + caption[1:] if caption else caption
    
    results.append({
        "file_name": rel_path,
        "text": caption
    })
    done_files.add(rel_path)
    
    # Checkpoint
    if len(results) % CHECKPOINT_EVERY == 0:
        with open(CHECKPOINT_PATH, "w") as f:
            json.dump(results, f)
        tqdm.write(f"  [Checkpoint: {len(results)}/{len(train_image_paths)}]")

print(f"\nDone! Captioned {len(results)} images - train split")

Captioning:   3%|▎         | 100/2975 [01:31<41:15,  1.16it/s]

  [Checkpoint: 100/2975]


Captioning:   7%|▋         | 200/2975 [02:56<38:44,  1.19it/s]

  [Checkpoint: 200/2975]


Captioning:  10%|█         | 300/2975 [04:23<38:30,  1.16it/s]

...

  [Checkpoint: 2800/2975]


Captioning:  97%|█████████▋| 2900/2975 [43:41<01:10,  1.07it/s]

  [Checkpoint: 2900/2975]


Captioning: 100%|██████████| 2975/2975 [44:49<00:00,  1.11it/s]


Done! Captioned 2975 images - train split


In [6]:
for img_path in tqdm(test_image_paths, desc="Captioning"):
    # Use relative path from the train directory as the identifier
    rel_path = os.path.relpath(img_path, os.path.dirname(TEST_DIR))
    
    if rel_path in done_files:
        continue
    
    image = Image.open(img_path).convert("RGB")
    inputs = processor(text=TASK, images=image, return_tensors="pt").to("cuda", torch.float16)
    
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1024,
            do_sample=False,
            num_beams=3
        )
    
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed = processor.post_process_generation(
        generated_text, task=TASK, image_size=(image.width, image.height)
    )
    caption = parsed[TASK]

    caption = re.sub(r'^The image shows\s+', '', caption, flags=re.IGNORECASE)
    caption = re.sub(r'^The image depicts\s+', '', caption, flags=re.IGNORECASE)
    caption = re.sub(r'^The image features\s+', '', caption, flags=re.IGNORECASE)
    caption = re.sub(r'^This image shows\s+', '', caption, flags=re.IGNORECASE)
    # Capitalize the first letter after stripping
    caption = caption[0].upper() + caption[1:] if caption else caption
    
    results.append({
        "file_name": rel_path,
        "text": caption
    })
    done_files.add(rel_path)
    
    # Checkpoint
    if len(results) % CHECKPOINT_EVERY == 0:
        with open(CHECKPOINT_PATH, "w") as f:
            json.dump(results, f)
        tqdm.write(f"  [Checkpoint: {len(results)}/{len(test_image_paths)}]")

print(f"\nDone! Captioned {len(results)} images - test split")

Captioning:   2%|▏         | 25/1525 [00:22<22:31,  1.11it/s]

  [Checkpoint: 3000/1525]


Captioning:   8%|▊         | 125/1525 [01:53<21:06,  1.11it/s]

  [Checkpoint: 3100/1525]


Captioning:  15%|█▍        | 225/1525 [03:21<20:57,  1.03it/s]

...

  [Checkpoint: 4300/1525]


Captioning:  93%|█████████▎| 1425/1525 [21:21<01:31,  1.10it/s]

  [Checkpoint: 4400/1525]


Captioning: 100%|██████████| 1525/1525 [22:51<00:00,  1.11it/s]

  [Checkpoint: 4500/1525]

Done! Captioned 4500 images - test split


In [7]:
for img_path in tqdm(val_image_paths, desc="Captioning"):
    # Use relative path from the train directory as the identifier
    rel_path = os.path.relpath(img_path, os.path.dirname(VAL_DIR))
    
    if rel_path in done_files:
        continue
    
    image = Image.open(img_path).convert("RGB")
    inputs = processor(text=TASK, images=image, return_tensors="pt").to("cuda", torch.float16)
    
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1024,
            do_sample=False,
            num_beams=3
        )
    
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed = processor.post_process_generation(
        generated_text, task=TASK, image_size=(image.width, image.height)
    )
    caption = parsed[TASK]

    caption = re.sub(r'^The image shows\s+', '', caption, flags=re.IGNORECASE)
    caption = re.sub(r'^The image depicts\s+', '', caption, flags=re.IGNORECASE)
    caption = re.sub(r'^The image features\s+', '', caption, flags=re.IGNORECASE)
    caption = re.sub(r'^This image shows\s+', '', caption, flags=re.IGNORECASE)
    # Capitalize the first letter after stripping
    caption = caption[0].upper() + caption[1:] if caption else caption
    
    results.append({
        "file_name": rel_path,
        "text": caption
    })
    done_files.add(rel_path)
    
    # Checkpoint
    if len(results) % CHECKPOINT_EVERY == 0:
        with open(CHECKPOINT_PATH, "w") as f:
            json.dump(results, f)
        tqdm.write(f"  [Checkpoint: {len(results)}/{len(val_image_paths)}]")

print(f"\nDone! Captioned {len(results)} images - validation split")

Captioning:  20%|██        | 100/500 [01:30<06:20,  1.05it/s]

  [Checkpoint: 4600/500]


Captioning:  40%|████      | 200/500 [03:00<04:53,  1.02it/s]

  [Checkpoint: 4700/500]


Captioning:  60%|██████    | 300/500 [04:28<03:01,  1.10it/s]

  [Checkpoint: 4800/500]


Captioning:  80%|████████  | 400/500 [05:58<01:31,  1.10it/s]

  [Checkpoint: 4900/500]


Captioning: 100%|██████████| 500/500 [07:27<00:00,  1.12it/s]

  [Checkpoint: 5000/500]

Done! Captioned 5000 images - validation split


In [8]:
# Save as JSONL (what diffusers LoRA training expects)
jsonl_path = os.path.join(OUTPUT_DIR, "metadata.jsonl")
with open(jsonl_path, "w", encoding="utf-8") as f:
    for entry in results:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

# Save as JSON (easy to inspect later)
json_path = os.path.join(OUTPUT_DIR, "captions.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

# Clean up checkpoint file
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)

print(f"Saved {len(results)} captions")
print(f"  JSONL: {jsonl_path}")
print(f"  JSON:  {json_path}")

Saved 5000 captions
  JSONL: /kaggle/working/metadata.jsonl
  JSON:  /kaggle/working/captions.json


In [9]:
import random

print("=" * 70)
print("SAMPLE CAPTIONS")
print("=" * 70)

for entry in random.sample(results, min(5, len(results))):
    print(f"\nFile: {entry['file_name']}")
    print(f"Caption: {entry['text']}")
    print("-" * 70)

# Basic statistics
caption_lengths = [len(entry["text"].split()) for entry in results]
print(f"\nCaption statistics:")
print(f"  Total captions: {len(results)}")
print(f"  Average length: {sum(caption_lengths)/len(caption_lengths):.0f} words")
print(f"  Shortest: {min(caption_lengths)} words")
print(f"  Longest: {max(caption_lengths)} words")

SAMPLE CAPTIONS

File: train/aachen/aachen_000070_000019_leftImg8bit.png
Caption: A city street with cars and buses driving down it. There are buildings with windows on either side of the street, trees lining the sides, and bicycles parked on the sidewalk. The sky is visible in the background.
----------------------------------------------------------------------

File: train/strasbourg/strasbourg_000001_002354_leftImg8bit.png
Caption: A man and woman walking down a street next to a car. The man is holding a bicycle and the woman is carrying a handbag. In the background, there is a building with windows and a door.
----------------------------------------------------------------------

File: test/berlin/berlin_000287_000019_leftImg8bit.png
Caption: A city street filled with lots of cars and people walking down it. There are buildings with windows on either side of the street, and awnings on the buildings. There is a traffic signal with a pole in the foreground, and in the background th